In [508]:
import pandas as pd
import numpy as np

In [509]:
data=pd.read_csv('D:\Bangalore House Price Predictor\data\Bengaluru_House_Data.csv')

In [510]:
data

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00
...,...,...,...,...,...,...,...,...,...
13315,Built-up Area,Ready To Move,Whitefield,5 Bedroom,ArsiaEx,3453,4.0,0.0,231.00
13316,Super built-up Area,Ready To Move,Richards Town,4 BHK,NaN,3600,5.0,NaN,400.00
13317,Built-up Area,Ready To Move,Raja Rajeshwari Nagar,2 BHK,Mahla T,1141,2.0,1.0,60.00
13318,Super built-up Area,18-Jun,Padmanabhanagar,4 BHK,SollyCl,4689,4.0,1.0,488.00


In [511]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13320 non-null  object 
 1   availability  13320 non-null  object 
 2   location      13319 non-null  object 
 3   size          13304 non-null  object 
 4   society       7818 non-null   object 
 5   total_sqft    13320 non-null  object 
 6   bath          13247 non-null  float64
 7   balcony       12711 non-null  float64
 8   price         13320 non-null  float64
dtypes: float64(3), object(6)
memory usage: 936.7+ KB


In [512]:
data.isnull().sum()

area_type          0
availability       0
location           1
size              16
society         5502
total_sqft         0
bath              73
balcony          609
price              0
dtype: int64

In [513]:
#total row->13320,  sociecty has 5502 missing values (too many), so we will drop it 
data.drop(columns=['society'], inplace=True)

In [514]:
data.isnull().sum()

area_type         0
availability      0
location          1
size             16
total_sqft        0
bath             73
balcony         609
price             0
dtype: int64

In [515]:
#location has 1 and size has 16 and bath has 73 missing values only so we can drop their rows easily
data.dropna(subset=['location','size','bath'], inplace=True)
#Balcony has 609 missing values, a bit higher, we'll replace them by the median
data['balcony'] = data['balcony'].fillna(data['balcony'].median())

In [516]:
#no null values
data.isnull().sum()

area_type       0
availability    0
location        0
size            0
total_sqft      0
bath            0
balcony         0
price           0
dtype: int64

In [517]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13246 entries, 0 to 13319
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13246 non-null  object 
 1   availability  13246 non-null  object 
 2   location      13246 non-null  object 
 3   size          13246 non-null  object 
 4   total_sqft    13246 non-null  object 
 5   bath          13246 non-null  float64
 6   balcony       13246 non-null  float64
 7   price         13246 non-null  float64
dtypes: float64(3), object(5)
memory usage: 931.4+ KB


In [518]:
data['area_type'].value_counts()

area_type
Super built-up  Area    8740
Built-up  Area          2410
Plot  Area              2009
Carpet  Area              87
Name: count, dtype: int64

In [519]:
data['availability'].value_counts()

availability
Ready To Move    10564
18-Dec             297
18-May             291
18-Apr             269
18-Aug             200
                 ...  
16-Oct               1
17-Jan               1
16-Nov               1
16-Jan               1
14-Jul               1
Name: count, Length: 80, dtype: int64

In [520]:
data.drop(columns=['area_type', 'availability'], inplace=True)

In [521]:
#size(object) to bhk(float), drop size
data['bhk'] = data['size'].str.split().str.get(0).astype(int)
data.drop(columns=['size'], inplace=True)

In [522]:
def convert_sqft(x):
    if '-' in x:
        a = x.split('-')
        return (float(a[0]) + float(a[1])) / 2
    try:
        return float(x)
    except:
        return None
#sqft(object with range to float)
data['total_sqft'] = data['total_sqft'].apply(convert_sqft)

In [523]:
data['total_sqft'].isna().sum()

np.int64(46)

In [524]:
data.dropna(subset=['total_sqft'], inplace=True)

In [525]:
#since we will one-hot encode this, it's better to mark the locations by 'other' which are appearing <=40
location_counts = data['location'].value_counts()
locations_less_than_0 = location_counts[location_counts <= 40]

data['location'] = data['location'].apply(
    lambda x: 'other' if x in locations_less_than_0 else x
)

In [526]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13200 entries, 0 to 13319
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   location    13200 non-null  object 
 1   total_sqft  13200 non-null  float64
 2   bath        13200 non-null  float64
 3   balcony     13200 non-null  float64
 4   price       13200 non-null  float64
 5   bhk         13200 non-null  int64  
dtypes: float64(4), int64(1), object(1)
memory usage: 721.9+ KB


In [527]:
data=data[data['total_sqft'] / data['bhk']>300]

In [528]:
# # remove houses with too small area per bhk
# data['sqft_per_bhk'] = data['total_sqft'] / data['bhk']
# data = data[data['sqft_per_bhk']>=300]
# # data.drop(columns=['sqft_per_bhk'], inplace=True)

# data

In [529]:
data['price_per_sqft'] = data['price'] * 100000 / data['total_sqft']

C:\Users\Joyesh\AppData\Local\Temp\ipykernel_2932\1709622629.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['price_per_sqft'] = data['price'] * 100000 / data['total_sqft']


In [530]:
data

,location,total_sqft,bath,balcony,price,bhk,price_per_sqft
0,Electronic City Phase II,1056.0,2.0,1.0,39.07,2,3699.810606
1,other,2600.0,5.0,3.0,120.00,4,4615.384615
2,Uttarahalli,1440.0,2.0,3.0,62.00,3,4305.555556
3,other,1521.0,3.0,1.0,95.00,3,6245.890861
4,Kothanur,1200.0,2.0,1.0,51.00,2,4250.000000
...,...,...,...,...,...,...,...
13315,Whitefield,3453.0,4.0,0.0,231.00,5,6689.834926
13316,other,3600.0,5.0,2.0,400.00,4,11111.111111
13317,Raja Rajeshwari Nagar,1141.0,2.0,1.0,60.00,2,5258.545136
13318,other,4689.0,4.0,1.0,488.00,4,10407.336319


In [531]:
#removing price_per_sqft based on location
def remove_pps_outliers(df):

    df_out = pd.DataFrame()

    for key, subdf in df.groupby('location'):

        mean = subdf.price_per_sqft.mean()
        std = subdf.price_per_sqft.std()

        reduced_df = subdf[
            (subdf.price_per_sqft > (mean - std)) &
            (subdf.price_per_sqft <= (mean + std))
        ]

        df_out = pd.concat([df_out, reduced_df], ignore_index=True)

    return df_out


data = remove_pps_outliers(data)

In [532]:
#removing bhk_outliers
def remove_bhk_outliers(df): #code

    exclude_indices = np.array([])

    for location, location_df in df.groupby('location'):

        bhk_stats = {}

        for bhk, bhk_df in location_df.groupby('bhk'):

            bhk_stats[bhk] = {
                'mean': bhk_df.price_per_sqft.mean(),
                'count': bhk_df.shape[0]
            }

        for bhk, bhk_df in location_df.groupby('bhk'):

            stats = bhk_stats.get(bhk - 1)

            if stats and stats['count'] > 5:

                exclude_indices = np.append(
                    exclude_indices,
                    bhk_df[bhk_df.price_per_sqft < stats['mean']].index.values
                )

    return df.drop(exclude_indices, axis='index')

data = remove_bhk_outliers(data)

In [533]:
#keeping it will cause data leakage
data.drop(columns=['price_per_sqft'], inplace=True)

In [534]:
#removing which has too many bathrooms
data = data[data['bath'] <= data['bhk'] * 1.3]

In [535]:
data = data[data['total_sqft'] < 12000]

In [536]:
data = pd.get_dummies(data, columns=['location'], drop_first=True)

In [537]:
data

,total_sqft,bath,balcony,price,bhk,location_8th Phase JP Nagar,location_9th Phase JP Nagar,location_Akshaya Nagar,location_Attibele,location_Balagere,...,location_Thigalarapalya,location_Uttarahalli,location_Varthur,location_Vidyaranyapura,location_Vijayanagar,location_Vittasandra,location_Whitefield,location_Yelahanka,location_Yeshwanthpur,location_other
0,1080.0,2.0,2.0,72.0,2,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,1270.0,2.0,2.0,93.0,2,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,1420.0,2.0,1.0,100.0,3,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,1850.0,3.0,1.0,150.0,3,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,1245.0,2.0,1.0,94.0,2,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10466,1805.0,3.0,3.0,134.0,3,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
10467,1715.0,3.0,3.0,112.0,3,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
10468,3600.0,5.0,2.0,400.0,4,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
10469,4689.0,4.0,1.0,488.0,4,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True


In [538]:
X = data.drop(columns=['price'])
y = data['price']

In [539]:
#train-test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [540]:
#scaling for regression models
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [541]:
# ===============================
# 1. Import Libraries
# ===============================

import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ===============================
# 2. Train Test Split
# ===============================

X = data.drop(columns=['price'])
y = data['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# ===============================
# 3. Scaling (for linear models)
# ===============================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# ===============================
# 4. Define Models
# ===============================

models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.01),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42)
}


# ===============================
# 5. Training Function
# ===============================

def train_and_evaluate(models, X_train, X_test, X_train_scaled, X_test_scaled, y_train, y_test):

    results = {}

    for name, model in models.items():

        # Linear models use scaled data
        if name in ["Linear Regression", "Ridge", "Lasso"]:
            model.fit(X_train_scaled, y_train)
            y_pred = model.predict(X_test_scaled)

        # Tree models use original data
        else:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)

        results[name] = {
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        }

    return results


# ===============================
# 6. Run Models
# ===============================

results = train_and_evaluate(
    models,
    X_train, X_test,
    X_train_scaled, X_test_scaled,
    y_train, y_test
)


# ===============================
# 7. Display Results
# ===============================

results_df = pd.DataFrame(results).T
results_df = results_df.sort_values(by="R2", ascending=False)

print(results_df)

                         MAE       RMSE        R2
Random Forest      14.055055  23.953127  0.911815
Linear Regression  15.961946  27.194677  0.886331
Ridge              15.963504  27.196709  0.886314
Lasso              15.986945  27.205481  0.886241
Decision Tree      15.965113  27.888472  0.880458


In [544]:
X_full = data.drop(columns=['price'])
y_full = data['price']

In [556]:
print("--"*25 + "X" +"--"*25)
print(X_full.info())
print("--"*25 + "y" +"--"*25)
print(y_full.info())

--------------------------------------------------X--------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
Index: 6145 entries, 0 to 10470
Data columns (total 77 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   total_sqft                         6145 non-null   float64
 1   bath                               6145 non-null   float64
 2   balcony                            6145 non-null   float64
 3   bhk                                6145 non-null   int64  
 4   location_8th Phase JP Nagar        6145 non-null   bool   
 5   location_9th Phase JP Nagar        6145 non-null   bool   
 6   location_Akshaya Nagar             6145 non-null   bool   
 7   location_Attibele                  6145 non-null   bool   
 8   location_Balagere                  6145 non-null   bool   
 9   location_Banashankari              6145 non-null   bool   
 10  location_Bannerghatta 

In [557]:
#winner model training on the full dataset
from sklearn.ensemble import RandomForestRegressor

# Initialize the winner model
final_model = RandomForestRegressor(random_state=42)

# Train on the full dataset
X_full = data.drop(columns=['price'])
y_full = data['price']

final_model.fit(X_full, y_full)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample